# Character-Level LSTM in PyTorch

## Import Libraries

In [19]:
## torchvision is not needed since there are no images
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

## Create the Text Corpus

In [20]:
text = """
Machine learning allows computers to learn patterns from data.
Neural networks are powerful models that can learn complex relationships.
PyTorch makes building deep learning models straightforward.
"""

## Find Every Unique Character

In [21]:
chars = sorted(list(set(text)))

## Create Translation Dictionaries

In [22]:
## Translate the characters to numbers so the model can understand them
char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for idx, char in enumerate(chars)}

## Set vocab size for the final number of possible outputs the model can predict
vocab_size = len(chars)

## Encode the text
encoded_text = [char_to_idx[c] for c in text]

## Choose a Sequence Length

In [23]:
## Set sequence length so model isn't fed entire data at once
sequence_length = 20

## Create Training Examples

In [24]:
inputs = []
targets = []

for i in range(len(encoded_text) - sequence_length):

    sequence = encoded_text[i:i + sequence_length]

    target = encoded_text[i + sequence_length]

    inputs.append(sequence)

    targets.append(target)


## Create Dataset from Examples

In [25]:
class TextDataset(Dataset):

    def __init__(self, inputs, targets):

        self.inputs = torch.tensor(inputs, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):

        return len(self.inputs)

    def __getitem__(self, idx):

        return self.inputs[idx], self.targets[idx]

## Create instanse of the dataset
dataset = TextDataset(inputs, targets)

## Create a DataLoader to handle batching and shuffling
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)



## Define the LSTM Model

In [26]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        ## Embedding layer to convert character indices to dense vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=32
        )    

        ## LSTM layer to process sequences of embeddings
        self.lstm = nn.LSTM(
            input_size=32,
            hidden_size=128,
            batch_first=True
        )

        ## Linear layer to map LSTM outputs to character probabilities
        self.fc = nn.Linear(
            128,
            vocab_size
        )

    def forward(self, x):

        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        x = self.fc(output[:, -1, :])

        return x 
    
model = CharLSTM(vocab_size)    

## Loss and Optimizer

In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training Loop

In [28]:
epochs = 20

for epoch in range(epochs):

    running_loss = 0.0

    for inputs, targets in train_loader:

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/20], Loss: 3.3637
Epoch [2/20], Loss: 3.2937
Epoch [3/20], Loss: 3.2070
Epoch [4/20], Loss: 3.0320
Epoch [5/20], Loss: 2.9441
Epoch [6/20], Loss: 2.8870
Epoch [7/20], Loss: 2.8099
Epoch [8/20], Loss: 2.7533
Epoch [9/20], Loss: 2.7092
Epoch [10/20], Loss: 2.5896
Epoch [11/20], Loss: 2.4949
Epoch [12/20], Loss: 2.4167
Epoch [13/20], Loss: 2.3141
Epoch [14/20], Loss: 2.2371
Epoch [15/20], Loss: 2.1595
Epoch [16/20], Loss: 2.0512
Epoch [17/20], Loss: 1.9785
Epoch [18/20], Loss: 1.8970
Epoch [19/20], Loss: 1.8340
Epoch [20/20], Loss: 1.7393


## Evaluating the Model

In [29]:
def generate_text(model, start_text, length):

    model.eval()

    generated = start_text

    with torch.no_grad():

        for _ in range(length):

            # Convert characters to indices
            input_seq = [
                char_to_idx[c]
                for c in generated[-sequence_length:]
            ]

            # Convert to tensor
            input_tensor = torch.tensor(
                input_seq,
                dtype=torch.long
            ).unsqueeze(0)


            # Get prediction
            output = model(input_tensor)


            # Get highest probability character
            predicted_idx = torch.argmax(output, dim=1).item()


            # Convert index back to character
            predicted_char = idx_to_char[predicted_idx]


            # Add prediction to text
            generated += predicted_char


    return generated

In [41]:
generated_text = generate_text(model, start_text="Machines", length=100)
print(generated_text)

## Due to the small dataset and limited training, the generated text may not be coherent. For better results, consider using a larger dataset and training for more epochs.

Machines moders tora tearn learn moders tora learn moders tora learn moders tora learn moders tora learn mod
